# 05 - TF-IDF Feature Engineering

Notebook ini membangun representasi teks menggunakan TF-IDF dari `master_dataset.csv`. Dataset yang digunakan sudah melewati tahap cleaning dan validasi pada notebook sebelumnya.

## Konsep TF-IDF

TF-IDF atau *Term Frequency-Inverse Document Frequency* adalah metode untuk memberi bobot pada kata berdasarkan kemunculannya di sebuah dokumen dan kelangkaannya pada seluruh kumpulan dokumen. Kata yang sering muncul pada satu tempat, tetapi tidak terlalu umum di semua tempat, akan mendapatkan bobot lebih tinggi.

## Mengapa TF-IDF Digunakan

Sistem ini menggunakan Content-Based Filtering. Oleh karena itu, setiap tempat perlu diubah menjadi fitur teks numerik agar dapat dibandingkan. TF-IDF dipilih karena sederhana, mudah dijelaskan, dan sesuai untuk Penulisan Ilmiah tanpa menggunakan deep learning atau model transformer.

In [ ]:
import pickle
import re
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

DATA_DIR = Path("../data/processed")
MASTER_FILE = DATA_DIR / "master_dataset.csv"
TFIDF_DATASET_FILE = DATA_DIR / "tfidf_dataset.csv"
TFIDF_MATRIX_FILE = DATA_DIR / "tfidf_matrix.pkl"
TFIDF_VECTORIZER_FILE = DATA_DIR / "tfidf_vectorizer.pkl"

TEXT_COLUMNS = ["name", "category", "subtypes", "address", "description"]
REQUIRED_COLUMNS = ["name", "category", "rating", "address", "subtypes", "type", "price_estimate"]

In [ ]:
# Load master dataset dengan validasi sederhana.
df = pd.read_csv(MASTER_FILE)
df.columns = df.columns.str.strip().str.lower()

missing_columns = sorted(set(REQUIRED_COLUMNS) - set(df.columns))
if missing_columns:
    raise ValueError(f"Kolom wajib belum tersedia: {missing_columns}")

# master_dataset saat ini tidak selalu memiliki description.
# Kolom ini dibuat kosong agar struktur TF-IDF tetap konsisten.
if "description" not in df.columns:
    df["description"] = ""

print("Dataset shape:", df.shape)
print("Type counts:")
print(df["type"].value_counts())

df[REQUIRED_COLUMNS].head()

## Proses Preprocessing

Preprocessing dibuat sederhana: teks diubah menjadi huruf kecil, tanda baca dibersihkan, spasi dinormalisasi, dan stopword dihapus. Notebook ini tidak menggunakan stemming, lemmatization, transformer, deep learning, atau SBERT agar implementasi tetap mudah dijelaskan.

In [ ]:
# Stopword sederhana: gabungan English stopword dan beberapa kata umum Bahasa Indonesia.
INDONESIAN_STOPWORDS = {
    "yang", "dan", "di", "ke", "dari", "untuk", "dengan", "atau", "pada",
    "ini", "itu", "ada", "dalam", "kota", "kabupaten", "kec", "jalan", "jl",
    "no", "daerah", "istimewa", "yogyakarta", "indonesia"
}
STOPWORDS = set(ENGLISH_STOP_WORDS).union(INDONESIAN_STOPWORDS)


def clean_text(value):
    """Membersihkan teks untuk input TF-IDF."""
    if pd.isna(value):
        return ""
    text = str(value).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    # Token yang mengandung angka seperti nomor jalan dan kode pos dihapus agar fitur lebih informatif.
    words = [
        word for word in text.split()
        if word not in STOPWORDS and len(word) > 1 and not re.search(r"\d", word)
    ]
    return " ".join(words)


def build_combined_text(row):
    """Menggabungkan atribut tempat menjadi satu profil teks."""
    raw_text = " ".join(str(row[column]) for column in TEXT_COLUMNS if column in row.index)
    return clean_text(raw_text)

In [ ]:
# Membuat combined_text sebagai dasar fitur TF-IDF.
df["combined_text"] = df.apply(build_combined_text, axis=1)

# Jika ada teks kosong, isi dengan nama tempat agar baris tetap bisa diproses.
df["combined_text"] = df["combined_text"].where(
    df["combined_text"].str.len() > 0,
    df["name"].apply(clean_text)
)

print("Combined text examples:")
df[["name", "type", "combined_text"]].head(10)

In [ ]:
# Membuat model TF-IDF.
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=1
)

tfidf_matrix = vectorizer.fit_transform(df["combined_text"])
feature_names = vectorizer.get_feature_names_out()

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Feature count:", len(feature_names))
print("Feature examples:", feature_names[:30])

In [ ]:
# Preview nilai TF-IDF agar mudah dipahami pada dokumentasi.
preview_rows = min(5, tfidf_matrix.shape[0])
preview_cols = min(10, tfidf_matrix.shape[1])

tfidf_preview = pd.DataFrame(
    tfidf_matrix[:preview_rows, :preview_cols].toarray(),
    columns=feature_names[:preview_cols]
)

print("Sample TF-IDF values:")
tfidf_preview

In [ ]:
# Validasi sederhana sebelum menyimpan output.
if tfidf_matrix.shape[0] != len(df):
    raise AssertionError("Jumlah baris TF-IDF tidak sama dengan jumlah baris dataset.")

if len(feature_names) == 0:
    raise AssertionError("Fitur TF-IDF kosong.")

with open(TFIDF_MATRIX_FILE, "wb") as file:
    pickle.dump(tfidf_matrix, file)

with open(TFIDF_VECTORIZER_FILE, "wb") as file:
    pickle.dump(vectorizer, file)

df.to_csv(TFIDF_DATASET_FILE, index=False)

print(f"Saved: {TFIDF_MATRIX_FILE}")
print(f"Saved: {TFIDF_VECTORIZER_FILE}")
print(f"Saved: {TFIDF_DATASET_FILE}")